# Suite No. 4: AMPA/GABA Matrix Optimization

**jaxfne** Suite No. 4: Two-level AGSDR + Adam matrix parameter optimization.

This notebook demonstrates optimizing  and  weight matrix
parameters using the  spec with  as the inner
differentiable refinement optimizer.

**Truth status:**  — computational scaffold only.
No biological calibration, no mechanism claim.

In [ ]:
import jaxfne as jtfne
import optax
import jax.numpy as jnp
import json


## 1. Build the Model

A small E/I network with 20 neurons (16 E, 4 I).

In [ ]:
cfg = (
    jtfne.configuration()
    .network(name="V1", kind="cortical_column", n=20, cell_types={"E": 0.8, "PV": 0.2})
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(domain="laminar_column", conductivity="proxy",
           boundary="mean_zero_neumann", gauge="mean_zero")
    .probe(name="probe", modes=["spikes", "V_m", "CSD"])
)
model = jtfne.construct(cfg)
print("Model constructed:", model.cfg.metadata["truth_mode"])

## 2. Define Objectives and Simulation

Group-rate targets: E group at 12 Hz, I group at 8 Hz.

In [ ]:
n_E = 16  # 80% of 20
n_I = 4   # 20% of 20
objective = jtfne.rate_targets(
    groups={"E": list(range(n_E)), "I": list(range(n_E, n_E + n_I))},
    targets_hz={"E": 12.0, "I": 8.0},
)
sim = jtfne.simulation(duration_ms=100.0, dt_ms=0.1, seed=0)
print("Objective:", objective.name, "| Simulation: 100 ms")

gAMPA_w_spec = jtfne.matrix_parameter(
    mask="excitatory_to_all",  # scale rows from E neurons
    bounds=(0.3, 3.0),
)
gGABA_w_spec = jtfne.matrix_parameter(
    mask="E_to_I",  # scale E→I synaptic entries
    bounds=(0.1, 5.0),
)
print("gAMPA_w spec:", gAMPA_w_spec)
print("gGABA_w spec:", gGABA_w_spec)

In [ ]:
gAMPA_w_spec = jtfne.matrix_parameter(
    mask="excitatory_to_all",  # scale rows from E neurons
    bounds=(0.3, 3.0),
)
print("gAMPA_w spec:", gAMPA_w_spec)

## 4. Two-Level AGSDR + Adam Optimizer

OUTER: AGSDR proposes multiplicative scale candidates.  
INNER:  refines each candidate using a differentiable
soft-rate sigmoid surrogate loss ( gradient steps).

In [ ]:
optimizer = jtfne.agsdr(
    parameters={
        "gAMPA_w": gAMPA_w_spec,
        "gGABA_w": gGABA_w_spec,
    },
    generations=4,
    population_size=4,
    inner_optimizer=optax.adam(learning_rate=1e-2),
    inner_steps=5,
    seed=42,
)
print("Optimizer:", optimizer.to_dict()["optimizer_class"])
print("Parameters:", list(optimizer.parameters.keys()))
print("Inner optimizer metadata:", json.dumps(optimizer.to_dict()["inner_optimizer"], indent=2))

## 5. Run Matrix Optimization

The two-level loop: AGSDR outer + Adam inner surrogate + real-objective scoring.

In [ ]:
result = model.tune(
    objectives=objective,
    optimizer=optimizer,
    simulation=sim,
)
print("Best score:", result.best_score)
print("Best parameters:", result.best_parameters)
print("Tuning path:", result.summary.get("tuning_path"))

## 6. Inspect and Validate Result

The  contains the tuned weight matrix.  
All output is JSON-safe; no Optax objects are serialized.

In [ ]:
# Verify JSON safety of summary
json_str = json.dumps(result.summary, allow_nan=False)
assert isinstance(json_str, str), "Summary must be JSON-safe"

# Verify both parameters are optimized and are 2D matrices
assert "gAMPA_w" in result.best_parameters, "gAMPA_w must be optimized"
assert "gGABA_w" in result.best_parameters, "gGABA_w must be optimized"

gAMPA = jnp.asarray(result.best_parameters["gAMPA_w"])
gGABA = jnp.asarray(result.best_parameters["gGABA_w"])

assert gAMPA.ndim == 2, f"gAMPA_w must be 2D matrix, got {gAMPA.ndim}D"
assert gGABA.ndim == 2, f"gGABA_w must be 2D matrix, got {gGABA.ndim}D"
print(f"✓ gAMPA_w: shape {gAMPA.shape}, bounds [{gAMPA.min():.4f}, {gAMPA.max():.4f}]")
print(f"✓ gGABA_w: shape {gGABA.shape}, bounds [{gGABA.min():.4f}, {gGABA.max():.4f}]")

# Compare W matrices
orig_W = model.params["emitter"].W
tuned_W = result.model.params["emitter"].W
print("W changed:", not jnp.allclose(orig_W, tuned_W))
print("Truth mode:", result.summary.get("truth_mode"))

## 7. Summary

| Item | Value |
|------|-------|
| Optimizer | AGSDR outer + Adam inner |
| Parameter |  (matrix) |
| Outer loop | AGSDR stochastic delta rule |
| Inner loop |  + sigmoid spike surrogate |
| Final scoring |  (real objective) |
| Truth mode |  |

**Acceptance criteria met:**
-  tunes a matrix (not a scalar)
-  is JSON-safe (no Optax objects)
- Adam inner loop uses differentiable surrogate
- Final scoring uses real objective
- Backward compatible: scalar AGSDR unchanged
